Process all the parquet files from image parsing into one columnar `ideal_df` containing only night-time observations

In [1]:
from pathlib import Path
from dask import dataframe as dd

ideal_df_location = Path("../../data/ideal_df.parquet")
files = sorted(Path().glob("../../data/20*-*.parquet"))
len(files), files

(176,
 [PosixPath('../../data/2010-08.parquet'),
  PosixPath('../../data/2010-09.parquet'),
  PosixPath('../../data/2010-10.parquet'),
  PosixPath('../../data/2010-11.parquet'),
  PosixPath('../../data/2010-12.parquet'),
  PosixPath('../../data/2011-01.parquet'),
  PosixPath('../../data/2011-02.parquet'),
  PosixPath('../../data/2011-03.parquet'),
  PosixPath('../../data/2011-04.parquet'),
  PosixPath('../../data/2011-05.parquet'),
  PosixPath('../../data/2011-06.parquet'),
  PosixPath('../../data/2011-07.parquet'),
  PosixPath('../../data/2011-08.parquet'),
  PosixPath('../../data/2011-09.parquet'),
  PosixPath('../../data/2011-10.parquet'),
  PosixPath('../../data/2011-11.parquet'),
  PosixPath('../../data/2011-12.parquet'),
  PosixPath('../../data/2012-01.parquet'),
  PosixPath('../../data/2012-02.parquet'),
  PosixPath('../../data/2012-03.parquet'),
  PosixPath('../../data/2012-04.parquet'),
  PosixPath('../../data/2012-05.parquet'),
  PosixPath('../../data/2012-06.parquet'),
  Pos

In [2]:
dfs = [dd.read_parquet(str(f)).assign(source_file=f.name) for f in files]

df: dd.DataFrame = dd.concat(dfs)
df.describe

<bound method DataFrame.describe of Dask DataFrame Structure:
                   date    time exposure filename source_file
npartitions=176                                              
                 string  string   object   string      object
                    ...     ...      ...      ...         ...
...                 ...     ...      ...      ...         ...
                    ...     ...      ...      ...         ...
                    ...     ...      ...      ...         ...
Dask Name: concat, 353 expressions
Expr=Concat(frames=[Assign(frame=ReadParquetFSSpec(5542945)), Assign(frame=ReadParquetFSSpec(fa2a1c7)), Assign(frame=ReadParquetFSSpec(93288e4)), Assign(frame=ReadParquetFSSpec(7faf80a)), Assign(frame=ReadParquetFSSpec(496390c)), Assign(frame=ReadParquetFSSpec(d5b06ca)), Assign(frame=ReadParquetFSSpec(0f8dd49)), Assign(frame=ReadParquetFSSpec(1d715e9)), Assign(frame=ReadParquetFSSpec(d890623)), Assign(frame=ReadParquetFSSpec(431e1df)), Assign(frame=ReadParquetFSSpe

In [3]:
from allsky.validation import is_valid_record_series

invalid_rows = ~is_valid_record_series(df)
df = df[~invalid_rows]
df = df.assign(
    timestamp=dd.to_datetime(
        df["date"] + " " + df["time"],
        format="%Y/%m/%d %H:%M:%S",
    )
)

Remove rows for any daytime images, or nighttime images with a lit moon overhead

In [5]:
from allsky.astronomy import (
    get_times_from_dataframe,
    get_moon_phase,
    get_altaz,
    sun,
    moon,
    ASTRONOMICAL_TWILIGHT,
    NEWMOON_LIGHT_FRACTION,
)

times = get_times_from_dataframe(df)
times

/run/current-system/sw/lib/python3.13/site-packages/dask/dataframe/dask_expr/_collection.py:4399: UserWarning: 
You did not provide metadata, so Dask is running your function on a small dataset to guess output types. It is possible that Dask will guess incorrectly.
To provide an explicit output types or to silence this message, please provide the `meta=` keyword, as described in the apply function that you are using.
  Before: .apply(func)
  After:  .apply(func, meta=('timestamp', 'datetime64[ns, US/Eastern]'))

  warnings.warn(meta_warning(meta))


In [18]:
sun_alt, _ = get_altaz(sun, times)
moon_alt, _ = get_altaz(moon, times)
moonlit = get_moon_phase(times)

sun_alt, moon_alt, moonlit

(array([ 11.22487166,   9.9290376 ,   9.74253194, ..., -33.34500366,
        -33.55056635, -33.67001403], shape=(2856868,)),
 array([18.90451793, 17.64943829, 17.46790827, ..., 60.51236192,
        60.27075575, 60.12951202], shape=(2856868,)),
 array([0.05205806, 0.05229125, 0.05232509, ..., 0.6443312 , 0.64440571,
        0.64444923], shape=(2856868,)))

In [19]:
sundown_condition = sun_alt < ASTRONOMICAL_TWILIGHT
moondown_condition = moon_alt < ASTRONOMICAL_TWILIGHT
newmoon_condition = moonlit < NEWMOON_LIGHT_FRACTION

In [24]:
ideal_df = df.compute()[sundown_condition & (moondown_condition | newmoon_condition)]
ideal_df

,date,time,exposure,filename,source_file,timestamp
8,2010/08/11,22:38:00,2.7883,000021274,2010-08.parquet,2010-08-11 22:38:00
9,2010/08/11,22:39:17,2.4769,000021275,2010-08.parquet,2010-08-11 22:39:17
10,2010/08/11,22:40:17,2.4769,000021276,2010-08.parquet,2010-08-11 22:40:17
11,2010/08/11,22:41:34,2.4769,000021277,2010-08.parquet,2010-08-11 22:41:34
12,2010/08/11,22:42:34,2.4769,000021278,2010-08.parquet,2010-08-11 22:42:34
...,...,...,...,...,...,...
4,2026/03/25,05:40:08,32.0241,000422467,2026-03.parquet,2026-03-25 05:40:08
5,2026/03/25,05:40:41,32.0241,000422468,2026-03.parquet,2026-03-25 05:40:41
6,2026/03/25,05:41:34,36.3559,000422469,2026-03.parquet,2026-03-25 05:41:34
7,2026/03/25,05:42:11,36.3559,000422470,2026-03.parquet,2026-03-25 05:42:11


In [ ]:
ideal_df.to_parquet(ideal_df_location)